In [1]:
import os
import time
import math
import tqdm
import numpy as np
import torch
import torch.nn as nn
import torch.utils.data
import torch.nn.functional as F
from torchsummary import summary
from torchvision import transforms
from torch.utils.tensorboard import SummaryWriter
from torch.autograd.function import InplaceFunction, Function

import biosppy as bio

import shutil
import subprocess
from pathlib import Path
from datetime import datetime
from collections import deque

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.metrics import roc_curve, auc, confusion_matrix
from sklearn import preprocessing

%matplotlib notebook
import matplotlib.pyplot as plt

import sys
sys.path.append('../')

# Physionet Specific imports
from driver import load_challenge_data
from driver import get_classes

# compute_auc(labels, probabilities, num_classes, check_errors=True)
from evaluate_12ECG_score import compute_auc
# compute_beta_score(labels, output, beta, num_classes, check_errors=True)
from evaluate_12ECG_score import compute_beta_score

In [2]:
DATALOADER_WORKER = 0

base_path = Path('../')

input_path = base_path.joinpath('input_directory')
output_path = base_path.joinpath('output_directory')

print(input_path.absolute())

C:\Users\opt\projects\physionet2020\notebooks\..\input_directory


In [3]:
def progress(train_accu, train_loss, val_accu, val_loss):
    return "Train/Accu: {:.4f} Train/Loss: {:.4f} "\
            "Val/Accu: {:.4f} Val/Loss: {:.4f}"\
            .format(train_accu, train_loss, val_accu, val_loss)

In [4]:
input_files = []
for entry in input_path.iterdir():        
    if entry.name.endswith('mat'):
        input_files.append(entry.name)

classes = get_classes(str(input_path), input_files)

data = []
headers = []
for i, f in enumerate(input_files):
    ecg, header = load_challenge_data(os.path.join(str(input_path), input_files[i]))
    data.append(ecg)
    headers.append(header)

In [5]:
# Check if we always have 12 leads and if the data length is equal
length = []
leads = []
for d in data:
    length.append(len(d[0]))
    leads.append(len(d))
    
print(np.min(length), np.max(length))
print(np.unique(leads))

3000 72000
[12]


In [6]:
# Split into train validation and test set on file level
# TODO: Assuming that each set contains each set contains all classes
files_train, files_test_val = train_test_split(input_files, test_size=0.2, random_state=42)
files_test, files_val = train_test_split(files_test_val, test_size=0.5, random_state=42)

# Iterate over files and create X_test, X_val and X_test and their labels y accordingly
# Attention: To keep it simple for the first submission we select a length of 3000
X_train, y_train = [], []
X_val, y_val = [], []
X_test, y_test = [], []

# TODO: This is not nice, but should be enough for the first submission
def load_data(files):
    X, y = [], []
    for i, f in enumerate(files):
        ecg, header = load_challenge_data(os.path.join(str(input_path), f))
        ecg_length = len(ecg[0])
        ecg = (ecg - ecg.mean()) / np.sqrt(ecg.var() + 1e-8)
        x_temp = []
        for lead in ecg:
            x_temp.append(lead)
        X.append(x_temp)
        sz = []
        for lines in header:
            if lines.startswith('#Dx'):
                tmp = lines.split(': ')[1].split(',')
                for c in tmp:
                    sz.append(c.strip())
        y.append(sz)
    return np.asarray(X), y

def load_data(files):
    datas = []; labels = []
    for file in tqdm.tqdm(files):
        data, header = load_challenge_data(str(input_path.joinpath(file)))
        datas.append(data)
        sz = []
        for lines in header:
            if lines.startswith('#Dx'):
                tmp = lines.split(': ')[1].split(',')
                for c in tmp:
                    sz.append(c.strip())
        labels.append(sz)
    return datas, labels

In [7]:
X_train, y_train_r = load_data(files_train)
X_val, y_val_r = load_data(files_val)
X_test, y_test_r = load_data(files_test)

classes = []
for record in [*y_train_r, *y_val_r, *y_test_r]:
    classes.extend(record)
classes = list(np.unique(classes))
print(classes)

# Label encoder with multi-hot encoded labesl
label_encoder = preprocessing.MultiLabelBinarizer()
label_encoder.fit([classes])
y_train = label_encoder.transform(y_train_r)
y_val = label_encoder.transform(y_val_r)
y_test = label_encoder.transform(y_test_r)

100%|███████████████████████████████████████████████████████████████████████████████| 688/688 [00:00<00:00, 846.67it/s]

['AF', 'I-AVB', 'LBBB', 'Normal', 'PAC', 'PVC', 'RBBB', 'STD', 'STE']


In [8]:
print(len(np.unique(y_train_r)))
print('Training Set:', len(y_train), len(X_train))
print('Validation Set:', len(y_val), len(X_val))
print('Test Set:', len(y_test), len(X_test))

# Multi-Hot Label encoded
print('Label Encoded:\n', y_test[:3])

36
Training Set: 5501 5501
Validation Set: 688 688
Test Set: 688 688
Label Encoded:
 [[0 0 0 0 0 0 1 0 0]
 [0 0 0 1 0 0 0 0 0]
 [1 0 0 0 0 0 0 0 0]]


In [9]:
class AugmentTransform_PowerNoise:
    def __init__(self, sampling_frequency=512./4., noise=0.02, prob=0.5):
        self.sampling_frequency = sampling_frequency
        self.noise = noise
        self.prob = prob

    def __call__(self, data):
        if np.random.choice(2, p = (1.-self.prob, self.prob)) != 0:
            return data
        randomness = np.float32(2*(np.random.normal()-0.5))
        noise_signal = randomness * \
                       np.sin(50./self.sampling_frequency*2*np.pi*
                              np.arange(0, data.shape[0]), dtype=np.float32) + \
                       (1.-randomness) * \
                       np.cos(50. / self.sampling_frequency * 2 * np.pi *
                              np.arange(0, data.shape[0]), dtype=np.float32)
        return data + self.noise * np.array([noise_signal for i in range(data.shape[1])]).T

class AugmentTransform_RandomNoise:
    def __init__(self, mean=0., sigma=0.02, prob=0.5):
        self.mean = mean
        self.sigma = sigma
        self.prob = prob

    def __call__(self, data):
        if np.random.choice(2, p = (1.-self.prob, self.prob)) != 0:
            return data
        return data + np.random.normal(self.mean, self.sigma, data.shape).astype(np.float32)

class AugmentTransform_DropoutBursts:
    def __init__(self, value=0., length=100, prob=0.5):
        self.value = value
        self.length = int(length)
        self.prob = prob

    def __call__(self, data):
        if np.random.choice(2, p = (1.-self.prob, self.prob)) != 0:
            return data
        position = np.random.randint(data.shape[0])
        out_data = np.copy(data)
        out_data[position:position+self.length] = self.value
        return out_data

In [10]:
class PreprocessTransform_Merge:
    def __init__(self, length = 65000):
        self.length = length
        
    def __call__(self, data):
        output_data = np.empty((self.length, 12), dtype=np.float32)

        # If the raw data length is bigger
        if data.shape[1] > self.length:
            output_data[:, :] = data[:, int(data.shape[1]/2 - self.length/2) : 
                                     int(data.shape[1]/2 + self.length/2)].T
            return output_data

        # Otherwise continue with merging
        # Find the rpeaks in the original signal
        tmp_data = 20 * (data[0] - data[0].mean()) / np.sqrt(data[0].var() + 1e-8)
        rpeaks, = bio.signals.ecg.hamilton_segmenter(tmp_data, sampling_rate=500.0)

        dest_beg_idx = 0
        dest_end_idx = rpeaks[-1]
        output_data[dest_beg_idx:dest_end_idx, :] = data[:, 0:rpeaks[-1]].T
        while True:
            rand_rpeak_beg = np.random.choice(rpeaks[:-1])
            rand_rpeak_end = np.random.choice(rpeaks[rpeaks > rand_rpeak_beg])
            insert_length = rand_rpeak_end - rand_rpeak_beg

            # Update dest_beg_idx and dest_end_idxt
            dest_beg_idx = dest_end_idx
            if (dest_end_idx + insert_length > output_data.shape[0]):
                insert_length = output_data.shape[0] - dest_end_idx
            dest_end_idx = dest_end_idx + insert_length

            if insert_length == 0:
                break

            output_data[dest_beg_idx:dest_end_idx, :] = \
                data[:, rand_rpeak_beg:rand_rpeak_beg+insert_length].T
        return output_data

In [11]:
transform = PreprocessTransform_Merge()

for i, record in tqdm.tqdm(enumerate(X_train)):
    X_train[i] = transform(record)

for i, record in tqdm.tqdm(enumerate(X_test)):
    X_test[i] = transform(record)

for i, record in tqdm.tqdm(enumerate(X_val)):
    X_val[i] = transform(record)

X_train = np.asarray(X_train)
X_test = np.asarray(X_test)
X_val = np.asarray(X_val)

5501it [01:23, 65.73it/s]
688it [00:14, 47.09it/s]
688it [00:12, 54.42it/s]


In [12]:
print(X_train.shape, X_val.shape, X_test.shape)

(5501, 65000, 12) (688, 65000, 12) (688, 65000, 12)


In [13]:
# TODO: Add augmentation, preprocessing, split and channel merge
class PhysionetDataset(torch.utils.data.Dataset):
    def __init__(self, X, y, cached=True, data_length=3000):

        self.X = X
        self.y = y

        self.augment = transforms.Compose([AugmentTransform_PowerNoise(),
                                           AugmentTransform_RandomNoise(),
                                           AugmentTransform_DropoutBursts(),
                                           transforms.ToTensor()])
        
        self.ids = np.arange(0, len(self.X))

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, index):
        _x, _y = self._data_on_map_index(index)
        #print('X:', _x, 'Y:', _y)
        x = self.augment(_x)[0, :, :]
        x, y = (x - x.mean()) / np.sqrt(x.var() + 1e-5), torch.tensor(_y.astype(np.int64), dtype=torch.float)
        return x, y

    def _data_on_map_index(self, index):
        data = self.X[self.ids[index]]
        label = self.y[self.ids[index]]
        return data, label

In [14]:
dataset_train = PhysionetDataset(X_train, y_train)
dataset_val = PhysionetDataset(X_val, y_val)
dataset_test = PhysionetDataset(X_test, y_test)

print('Number of Records (Train):\t', len(dataset_train))
print('Number of Records (Validation):\t', len(dataset_val))
print('Number of Records (Test):\t', len(dataset_test))

data, label = dataset_train[0]
print('Data:', data.shape, '\nLabel', label)

Number of Records (Train):	 5501
Number of Records (Validation):	 688
Number of Records (Test):	 688
Data: torch.Size([65000, 12]) 
Label tensor([0., 0., 0., 0., 1., 0., 0., 0., 0.])


In [22]:
class PhysionetCNN(nn.Module):
    def __init__(self, n_classes):
        super(PhysionetCNN, self).__init__()
        self.conv1 = nn.Conv1d(12, 24, 38, stride=4, padding=1)
        self.bn1 = nn.BatchNorm1d(24)
        
        self.conv2 = nn.Conv1d(24, 30, 32, stride=4, padding=1)
        self.bn2 = nn.BatchNorm1d(30)

        self.conv3 = nn.Conv1d(30, 34, 24, stride=4, padding=1)
        self.bn3 = nn.BatchNorm1d(34)

        self.conv4 = nn.Conv1d(34, 30, 21, stride=4, padding=0)
        self.bn4 = nn.BatchNorm1d(30)

        self.conv5 = nn.Conv1d(30, 28, 16, stride=4, padding=0)
        self.bn5 = nn.BatchNorm1d(28)

        self.fc1 = nn.Linear(28*59, 256)
        self.lbn1 = nn.BatchNorm1d(256)
        self.fc2 = nn.Linear(256, 128)
        self.lbn2 = nn.BatchNorm1d(128)
        self.fc3 = nn.Linear(128, n_classes)

    def forward(self, x):
        x = F.relu(self.bn1(self.conv1(x)))
        x = F.relu(self.bn2(self.conv2(x)))
        x = F.relu(self.bn3(self.conv3(x)))
        x = F.relu(self.bn4(self.conv4(x)))
        x = F.relu(self.bn5(self.conv5(x)))
        x = x.view(-1, 28*59)
        x = self.lbn1(F.relu(self.fc1(x)))
        x = self.lbn2(F.relu(self.fc2(x)))
        x = self.fc3(x)
        return x

In [23]:
# First pass through network
x, y = dataset_train[0]
net = PhysionetCNN(len(label_encoder.classes_))
net.eval()
net(x[None, :, :].transpose(1, 2))

tensor([[-0.0402,  0.0308, -0.0793,  0.0414, -0.1003, -0.0828,  0.0441, -0.0093,
          0.0288]], grad_fn=<AddmmBackward>)

In [24]:
# show network
print(net)
summary(net, x.transpose(0, 1).shape, device="cpu")

PhysionetCNN(
  (conv1): Conv1d(12, 24, kernel_size=(38,), stride=(4,), padding=(1,))
  (bn1): BatchNorm1d(24, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (conv2): Conv1d(24, 30, kernel_size=(32,), stride=(4,), padding=(1,))
  (bn2): BatchNorm1d(30, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (conv3): Conv1d(30, 34, kernel_size=(24,), stride=(4,), padding=(1,))
  (bn3): BatchNorm1d(34, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (conv4): Conv1d(34, 30, kernel_size=(21,), stride=(4,))
  (bn4): BatchNorm1d(30, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (conv5): Conv1d(30, 28, kernel_size=(16,), stride=(4,))
  (bn5): BatchNorm1d(28, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (fc1): Linear(in_features=1652, out_features=256, bias=True)
  (lbn1): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (fc2): Linear(in_features=256, out_features=128, bias

In [25]:
dataloader_train = torch.utils.data.DataLoader(dataset_train, batch_size=256,
                                               shuffle=True, num_workers=DATALOADER_WORKER,
                                               pin_memory=True)
dataloader_val = torch.utils.data.DataLoader(dataset_val, batch_size=100,
                                             shuffle=True, num_workers=DATALOADER_WORKER,
                                             pin_memory=True)

In [29]:
class PhysionetModel:
    def __init__(self, dataloader_train, dataloader_val, base_path, network,
                 learning_rate=0.05, sgd_momentum=0.9, n_classes=9,
                 weight_decay=0.00001, optimizer_name='sgd', scheduler=True):

        self.dataloader_train = dataloader_train
        self.dataloader_val = dataloader_val
        self.base_path = base_path
        self.model_path = self.base_path.joinpath('models')

        self.best_accuracy = 0.
        self.best_net_name = ''
        self.best_epoch = 0.

        # Initialize model
        self.net = network

        # Training on gpu
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        print('Using:', self.device)
        self.net = self.net.to(self.device)

        self.weight_decay = weight_decay
        self.learning_rate = learning_rate
        self.sgd_momentum = sgd_momentum

        self.optimizer = torch.optim.SGD(
            self.net.parameters(),
            lr=self.learning_rate,
            momentum=self.sgd_momentum,
            weight_decay=self.weight_decay,
            nesterov=True)

        self.scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            self.optimizer, patience=20, factor=0.5, min_lr=0.000001)

        self.criterion = nn.BCEWithLogitsLoss()
        #self.criterion = nn.NLLLoss()
        self.m = nn.Sigmoid()

    def __del__(self):
        print('Training Finished -> Closing Model with accuracy', self.best_accuracy)
        # log hyper parameter
        if self.best_accuracy != 0.:
            best_file = self.model_path.joinpath(self.best_net_name)
            if not best_file.exists():
                print('Stop Logging: Best File does not exist:', best_file)
                return

            # Load the best model during training
            self.load_network(self.best_net_name)
            
            print('Computing Metric for', best_file)
            
            # Get the predictions
            y_true, y_prob, y_pred = self.evaluate()
            # Print classification report
            crs = classification_report(y_true, y_pred, output_dict=False)
            print(crs)
            
            auroc, auprc, accuracy, f_measure, f_beta, g_beta = self.calculate_metrics(y_pred, y_prob, y_true)
            
            print('\nauroc:', auroc,
                  '\nauprc:', auprc,
                  '\naccuracy:', accuracy,
                  '\nf_measure:', f_measure,
                  '\nf_beta:', f_beta,
                  '\ng_beta:', g_beta,)
            

    def calculate_accuracy(self, output, target):
        beta = 2
        n_classes = 9

        y_pred = np.zeros(output.shape)
        for k in range(0, output.shape[0]):
            y_pred[k, output[k].argmax()] = 1.
        accuracy, f_measure, f_beta, g_beta = compute_beta_score(
            target, output,
            beta, n_classes, check_errors=True)
        
        return accuracy

            
    def calculate_metrics(self, output, output_prob, target):
        n_classes = 9
        beta = 2
        
        auroc, auprc = compute_auc(target,
                                   output_prob,
                                   n_classes, check_errors=True)
        
        accuracy, f_measure, f_beta, g_beta = compute_beta_score(
            target, output,
            beta, n_classes, check_errors=True)

        return auroc, auprc, accuracy, f_measure, f_beta, g_beta

    def train(self, epoch):
        self.net = self.net.train()

        avg_loss, avg_accu, cnt = 0, 0, 0
        for batch_idx, (data, target) in enumerate(self.dataloader_train):
            data = data.to(self.device).transpose(1, 2)
            target = target.to(self.device)
                
            # forward
            output = self.net(data)

            # backward
            self.optimizer.zero_grad()
            #loss = self.criterion(self.m(output), target)
            loss = self.criterion(output, target)
            
            loss.backward()
            self.optimizer.step()
            
            # measure accuracy on batch
            avg_accu += self.calculate_accuracy(self.m(output).detach().cpu().numpy(),
                                                target.detach().cpu().numpy())
            avg_loss += loss
            cnt += 1

        return avg_accu/cnt, avg_loss/cnt

    def validate(self, epoch):
        self.net = self.net.eval()

        avg_loss, avg_accu, cnt = 0, 0, 0
        with torch.no_grad():
            for batch_idx, (data, target) in enumerate(self.dataloader_val):
                data = data.to(self.device).transpose(1, 2)
                target = target.to(self.device)
                
                # forward
                output = self.net(data)

                # loss / accuracy
                avg_loss += self.criterion(output, target)
                avg_accu += self.calculate_accuracy(self.m(output).detach().cpu().numpy(),
                                                    target.detach().cpu().numpy())
                cnt += 1

        # Learning Rate reduction
        if self.scheduler:
            self.scheduler.step(avg_loss/cnt)
            
        if avg_accu/cnt > self.best_accuracy:
            self.best_accuracy = avg_accu/cnt
            self.best_net_name = self.save_network("{:05d}.pth".format(epoch))
            self.best_epoch = epoch

        return avg_accu/cnt, avg_loss

    def evaluate(self):
        self.net = self.net.eval()
        val_dataset = self.dataloader_val.dataset
        y_true = np.zeros([len(self.dataloader_val.dataset), 9], dtype=np.float32)
        y_prob = np.zeros([len(self.dataloader_val.dataset), 9], dtype=np.float32)
        y_pred = np.zeros([len(self.dataloader_val.dataset), 9], dtype=np.float32)

        with torch.no_grad():
            for i in range(0, len(val_dataset)):
                data, target = val_dataset[i]
                data = data.to(self.device).view(-1, *data.shape).transpose(1, 2)
                target = target.to(self.device)
                output = self.m(self.net(data))
                
                # Todo: Pfusch
                #y_pred[i, output.detach().cpu().numpy().flatten() > 0.5] = 1.
                y_pred[i, output.detach().cpu().numpy().argmax()] = 1.
                
                y_true[i, :] = target.detach().cpu().numpy()
                y_prob[i, :] = output.detach().cpu().numpy()
                
        return y_true, y_prob, y_pred
    
    def save_network(self, name):
        model_file_name = self.model_path.joinpath(name)
        torch.save(self.net.state_dict(), model_file_name)
        return name
        
    def load_network(self, name):
        model_file_name = self.model_path.joinpath(name)
        self.net.load_state_dict(torch.load(model_file_name))

In [30]:
network = PhysionetCNN(len(label_encoder.classes_))
model = PhysionetModel(dataloader_train, dataloader_val, Path('./'), network)

Using: cuda


In [ ]:
epochs = 2048
t = tqdm.tqdm(range(0, epochs), total=epochs, leave=True,
            desc=progress(0, 0, 0, 0))
for epoch in t:
    train_accu, train_loss = model.train(epoch)
    val_accu, val_loss = model.validate(epoch)
    t.set_description(progress(train_accu, train_loss, val_accu, val_loss))



Train/Accu: 0.0000 Train/Loss: 0.0000 Val/Accu: 0.0000 Val/Loss: 0.0000:   0%|                | 0/2048 [00:00<?, ?it/s]

Train/Accu: 0.1150 Train/Loss: 0.6438 Val/Accu: 0.1154 Val/Loss: 3.6511:   0%|                | 0/2048 [03:56<?, ?it/s]

Train/Accu: 0.1150 Train/Loss: 0.6438 Val/Accu: 0.1154 Val/Loss: 3.6511:   0%|   | 1/2048 [03:56<134:24:42, 236.39s/it]

Train/Accu: 0.1150 Train/Loss: 0.4230 Val/Accu: 0.1154 Val/Loss: 2.4077:   0%|   | 1/2048 [08:05<134:24:42, 236.39s/it]

Train/Accu: 0.1150 Train/Loss: 0.4230 Val/Accu: 0.1154 Val/Loss: 2.4077:   0%|   | 2/2048 [08:05<136:32:00, 240.24s/it]

Train/Accu: 0.1150 Train/Loss: 0.2778 Val/Accu: 0.1154 Val/Loss: 1.8512:   0%|   | 2/2048 [11:54<136:32:00, 240.24s/it]

Train/Accu: 0.1150 Train/Loss: 0.2778 Val/Accu: 0.1154 Val/Loss: 1.8512:   0%|   | 3/2048 [11:54<134:33:06, 236.86s/it]

Train/Accu: 0.1150 Train/Loss: 0.2016 Val/Accu: 0.1154 Val/Loss: 1.8906:   0%|   | 3/2048 [15:38<134:33:06, 236.86s/it]

Train/Accu: 0.1150 Train/Loss:

Training Finished -> Closing Model with accuracy 0.11552259611208068
Computing Metric for models\00098.pth
              precision    recall  f1-score   support

           0       0.80      0.80      0.80       121
           1       0.80      0.70      0.75        61
           2       0.90      0.76      0.83        25
           3       0.64      0.61      0.62        95
           4       0.22      0.25      0.23        53
           5       0.69      0.47      0.56        72
           6       0.91      0.84      0.87       199
           7       0.66      0.66      0.66        92
           8       0.44      0.33      0.38        21

   micro avg       0.73      0.68      0.70       739
   macro avg       0.67      0.60      0.63       739
weighted avg       0.73      0.68      0.70       739
 samples avg       0.73      0.69      0.70       739


auroc: 0.6959305944854215 
auprc: 0.46371735367335426 
accuracy: 0.9326624737945493 
f_measure: 0.6391322084065112 
f_beta: 0.6268788



Train/Accu: 0.1150 Train/Loss: 0.0107 Val/Accu: 0.1154 Val/Loss: 1.7326:   1%|  | 12/2048 [50:23<132:27:45, 234.22s/it]

Train/Accu: 0.1150 Train/Loss: 0.0107 Val/Accu: 0.1154 Val/Loss: 1.7326:   1%|  | 13/2048 [50:23<134:14:34, 237.48s/it]

Train/Accu: 0.1150 Train/Loss: 0.0089 Val/Accu: 0.1154 Val/Loss: 1.7304:   1%|  | 13/2048 [55:05<134:14:34, 237.48s/it]

Train/Accu: 0.1150 Train/Loss: 0.0089 Val/Accu: 0.1154 Val/Loss: 1.7304:   1%|  | 14/2048 [55:05<141:40:31, 250.75s/it]

Train/Accu: 0.1150 Train/Loss: 0.0074 Val/Accu: 0.1154 Val/Loss: 1.7569:   1%|  | 14/2048 [58:59<141:40:31, 250.75s/it]

Train/Accu: 0.1150 Train/Loss: 0.0074 Val/Accu: 0.1154 Val/Loss: 1.7569:   1%|  | 15/2048 [58:59<138:52:32, 245.92s/it]

Train/Accu: 0.1150 Train/Loss: 0.0063 Val/Accu: 0.1154 Val/Loss: 1.7888:   1%| | 15/2048 [1:02:44<138:52:32, 245.92s/it

Train/Accu: 0.1150 Train/Loss: 0.0063 Val/Accu: 0.1154 Val/Loss: 1.7888:   1%| | 16/2048 [1:02:44<135:18:06, 239.71s/it

Train/Accu: 0.1150 Train/Loss:

In [ ]:
del model
del network

In [ ]:
# Helper functions to find networks
def compute_output_length(input_length, padding, dilation, kernel_size, stride):
    return (input_length + 2 * padding - dilation * (kernel_size - 1) - 1) / stride + 1

def debug_network_dimensions(input_shape, conv_channels, conv_kernels,
                             conv_strides, conv_paddings, conv_dilations,
                             linear_features, show_weights=False, **kwargs):

    c_w = lambda c_in_size, kernel_size: np.sqrt(1/(c_in_size*kernel_size))
    l_w = lambda num_features : np.sqrt(1/num_features)

    input_length = input_shape
    channel_in = 1.
    for idx in range(len(conv_channels)):
        output_length = compute_output_length(input_length,
                                              conv_paddings[idx],
                                              conv_dilations[idx],
                                              conv_kernels[idx],
                                              conv_strides[idx])
        print('Layer: {:1d}'.format(idx),
            'Input Length: {:5d}'.format(input_length),
            'Output Length: {:8.3f}'.format(output_length),
            'Output Channels: {:2d}'.format(conv_channels[idx]),
            '(Kernel Size: {:2d}'.format(conv_kernels[idx]),
            'Padding: {:1d}'.format(conv_paddings[idx]),
            'Stride: {:1d})'.format(conv_strides[idx]))
        
        if show_weights:
            print('\t --> Suggested weights:', c_w(channel_in, conv_kernels[idx]))
        input_length = int(output_length)
        channel_in = conv_channels[idx]

    linear_bias = [True, True]
    linear_features.append(2)

    print('Conv Output Features: {:8.3f}'.format(output_length * conv_channels[idx]))
    for idx in range(len(linear_features) - 1):
        print('Layer: {:1d}'.format(idx + len(conv_channels) -1),
            'Input Length: {:5d}'.format(linear_features[idx]),
            'Output Length: {:4d}'.format(linear_features[idx+1]))
        
        if show_weights:
            print('\t --> Suggested weights:', l_w(linear_features[idx])) 

In [ ]:
nparam = {'net_name' : 'big',
          'conv_channels' : [4, 6, 8, 10, 12],
          'conv_kernels' : [38, 32, 24, 21, 16],
          'conv_strides' : [4, 4, 4, 4, 4],
          'conv_paddings' : [1, 1, 1, 0, 0],
          'conv_dilations' : [1, 1, 1, 1, 1],
          'linear_features' : [59*12, 256, 128]}

print('#####', nparam['net_name'])
debug_network_dimensions(65000, **nparam, show_weights=True) 